# Saudi Hip Fracture Demand to 2030 — Notebook 2: Model

Notebook 1 produced clean, verified tables (55 gates passed). This notebook turns them into the 2030 estimate.

## Step 1 – Compare my method with the paper's published 2015 estimate (2,949)
The paper estimated **2,949 hip fractures in people aged 50+ in 2015**, by applying its rates to the UN population. I repeat the same calculation with the current UN population (2024 revision) and compare.

Two ways of treating ages above 80 are tested, because the paper's table stops at "80+" while its text says rates keep rising up to age 90:
- **Flat:** every age 80+ gets the printed 80+ rate (exactly as printed)
- **Trend:** above 85, the rate keeps rising at the same yearly pace as between the 70–74 group and the 80+ group

Any remaining gap is **documented, not forced**: the paper used the UN 2019 population revision, which cannot be re-downloaded, while this study uses the 2024 revision. The ratio between the published 2,949 and my estimate becomes a **calibration factor** for a high scenario, so the final range covers both my estimate and the published one.

**Gates:** clean files load with the same rows as Notebook 1 · my calculation adds up across age groups · the trend rates are never below the flat rates · the calibration factor is a plausible size (between 1 and 2).

In [8]:
# Starting: I connect Google Drive and set up the same folders and gate as in Notebook 1
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

BASE  = '/content/drive/MyDrive/hip-fracture-saudi'
CLEAN = f'{BASE}/data/clean'
FIG   = f'{BASE}/figures'

GATE_LOG = []
def gate(condition, message):
    if not condition:
        raise AssertionError('GATE FAILED: ' + message)
    GATE_LOG.append(message)
    print('GATE PASSED:', message)

# Now I load the clean tables from Notebook 1 and confirm they are complete
pop    = pd.read_csv(f'{CLEAN}/pop2022_by_age_sex_nationality.csv')
rates  = pd.read_csv(f'{CLEAN}/hipfracture_rates_by_group.csv')
un     = pd.read_csv(f'{CLEAN}/un_population_2015_2030.csv')
proj   = pd.read_csv(f'{CLEAN}/published_projections.csv')
gate(len(pop) == 444 and len(rates) == 16 and len(un) == 672 and len(proj) == 15,
     'clean files loaded with the same rows as Notebook 1 (444 / 16 / 672 / 15)')

# Now I build a rate for every 5-year age group from 50 to 100+, in two versions: flat and trend
def rate_table(method):
    out = []
    for sex in ['male', 'female']:
        r = rates[rates['sex'] == sex].set_index('age_start')['rate_per_100k']
        slope = np.log(r[80] / r[70]) / 10          # yearly rise between the 70-74 group and the 80+ group
        for start in range(50, 105, 5):
            if start < 80:
                value = r[start]
            elif method == 'flat' or start == 80:
                value = r[80]                       # printed 80+ rate
            else:
                value = r[80] * np.exp(slope * (start - 80))   # keeps rising along the trend
            out.append({'sex': sex, 'age_start': start, 'rate_per_100k': value, 'method': method})
    return pd.DataFrame(out)

rate_versions = pd.concat([rate_table('flat'), rate_table('trend')], ignore_index=True)
flat  = rate_versions[rate_versions['method'] == 'flat'].reset_index(drop=True)
trend = rate_versions[rate_versions['method'] == 'trend'].reset_index(drop=True)
gate((trend['rate_per_100k'] >= flat['rate_per_100k'] - 1e-9).all(), 'trend rates are never below the flat rates')

# Now I apply each version to the UN 2015 population aged 50+, exactly as the paper did
un50 = un[(un['year'] == 2015) & (un['age_start'] >= 50)]
results = {}
for method, table in [('flat', flat), ('trend', trend)]:
    m = un50.merge(table, on=['sex', 'age_start'])
    gate(len(m) == len(un50), f'{method}: every UN age group 50+ found its rate ({len(m)} groups)')
    m['fractures'] = m['population'] * m['rate_per_100k'] / 100_000
    by_sex = m.groupby('sex')['fractures'].sum()
    gate(abs(by_sex.sum() - m['fractures'].sum()) < 1e-6, f'{method}: men + women add up to the total')
    results[method] = by_sex

# Now I compare with the paper's published 2015 numbers
pub = proj[proj['year'] == 2015].set_index('sex')['hip_fractures']
compare = pd.DataFrame({
    'Paper (UN 2019 revision)': [pub['Men'], pub['Women'], pub['Men and women']],
    'Mine - flat 80+':  [results['flat']['male'],  results['flat']['female'],  results['flat'].sum()],
    'Mine - trend 80+': [results['trend']['male'], results['trend']['female'], results['trend'].sum()],
}, index=['Men', 'Women', 'Total']).round(0)
compare['Gap flat %']  = ((compare['Mine - flat 80+']  / compare['Paper (UN 2019 revision)'] - 1) * 100).round(1)
compare['Gap trend %'] = ((compare['Mine - trend 80+'] / compare['Paper (UN 2019 revision)'] - 1) * 100).round(1)
print('\n2015 hip fractures, age 50+:')
print(compare.to_string())

# Finally I calculate the calibration factor for the high scenario and save everything for the next steps
calibration = pub['Men and women'] / results['trend'].sum()
gate(1 <= calibration <= 2, f'calibration factor is a plausible size: {calibration:.3f}')
rate_versions.to_csv(f'{CLEAN}/rates_flat_and_trend.csv', index=False)
compare.to_csv(f'{CLEAN}/comparison_2015_with_paper.csv')
pd.DataFrame({'calibration_factor': [calibration]}).to_csv(f'{CLEAN}/calibration_factor.csv', index=False)
print(f'\nHigh scenario will multiply the trend estimate by {calibration:.3f} so 2015 matches the published 2,949.')
print(f'\nStep 1 complete: {len(GATE_LOG)} gates passed, 0 failed')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GATE PASSED: clean files loaded with the same rows as Notebook 1 (444 / 16 / 672 / 15)
GATE PASSED: trend rates are never below the flat rates
GATE PASSED: flat: every UN age group 50+ found its rate (22 groups)
GATE PASSED: flat: men + women add up to the total
GATE PASSED: trend: every UN age group 50+ found its rate (22 groups)
GATE PASSED: trend: men + women add up to the total

2015 hip fractures, age 50+:
       Paper (UN 2019 revision)  Mine - flat 80+  Mine - trend 80+  Gap flat %  Gap trend %
Men                        1416           1105.0            1200.0       -22.0        -15.3
Women                      1533            992.0            1075.0       -35.3        -29.9
Total                      2949           2097.0            2274.0       -28.9        -22.9
GATE PASSED: calibration factor is a plausible size: 1.297

High scenario will multiply 

## Step 2 – Survival by age from the UN life table, and a backtest that proves the ageing method
To age the population from 2022 to 2030, I need the share of people at each age who survive one more year. The UN life table gives **l(x)**: out of 100,000 people born, how many are still alive at exact age x. One-year survival is then **l(x+1) ÷ l(x)**, for each sex and each calendar year.

**Backtest:** before trusting the method for 2030, I test it on years where the answer is known. I take the Saudi population by 5-year age group in **2017** (GASTAT), age every group forward 5 years using the life table, and compare with the **real 2022** population (GASTAT). If the method is sound, each predicted group should be within 3% of the real one.

**Assumptions (stated openly):**
- the UN life table covers everyone living in Saudi Arabia; it is applied to Saudi citizens
- Saudi citizens hardly migrate in or out, so survival is the only change between ages
- for the open group 65+, the age mix inside the group is taken from the UN 2017 population

**Gates:** the life table is complete (2,828 rows) and starts at 100,000 · survivors only decrease with age · every survival share is between 0 and 1 · **backtest: every predicted 2022 age group within ±3% of the real figure**.

In [9]:
# Starting: I load the raw UN life table (survivors to each exact age) and check it is complete
RAW = f'{BASE}/data/raw'
lt_raw = pd.read_csv(f'{RAW}/un_wpp_saudi_lifetable_survivors_2017_2030.csv')
gate(len(lt_raw) == 14 * 2 * 101, f'life table has {len(lt_raw):,} rows = 14 years x 2 sexes x 101 ages')
gate((lt_raw.loc[lt_raw['AgeStart'] == 0, 'Value'] == 100_000).all(), 'every life table starts at 100,000 at birth')

lt = (lt_raw.rename(columns={'Time': 'year', 'Sex': 'sex', 'AgeStart': 'age', 'Value': 'lx'})
            [['year', 'sex', 'age', 'lx']]
            .assign(sex=lambda d: d['sex'].str.lower())
            .sort_values(['year', 'sex', 'age'])
            .reset_index(drop=True))
gate(lt.groupby(['year', 'sex'])['lx'].apply(lambda s: s.is_monotonic_decreasing).all(),
     'survivors only decrease with age, in every year and for both sexes')

# Now I turn survivors into one-year survival: share of people aged x who reach age x+1
lt['lx_next'] = lt.groupby(['year', 'sex'])['lx'].shift(-1)
lt['survival'] = lt['lx_next'] / lt['lx']
# the last age is the open group 100+: I use the survival of age 99 (people aged 100+ stay in 100+)
last = lt['age'] == 100
lt.loc[last, 'survival'] = lt.loc[lt['age'] == 99, 'survival'].values
surv = lt[['year', 'sex', 'age', 'survival']]
gate(surv['survival'].between(0, 1).all() and surv['survival'].notna().all(),
     f'every one-year survival share is between 0 and 1 ({len(surv):,} values)')
surv.to_csv(f'{CLEAN}/survival_by_age_sex_year.csv', index=False)

# Now I write a helper: chance that someone aged `age` in `start_year` is still alive 5 years later
S = surv.set_index(['year', 'sex', 'age'])['survival']
def survive_5y(age, start_year):
    p = 1.0
    for k in range(5):
        a = min(age + k, 100)
        p *= (S[(start_year + k, 'male', a)] + S[(start_year + k, 'female', a)]) / 2   # both sexes together
    return p

def group_survival(start, ages):
    # average 5-year survival over the single ages in a 5-year group
    return np.mean([survive_5y(a, 2017) for a in ages])

# Now the BACKTEST: age the real 2017 Saudi population forward 5 years and compare with the real 2022 population
glmm = pd.read_csv(f'{RAW}/manual_tables/glmm_saudi_pop_agegroup_2010_2022.csv')
g = glmm[glmm['age_group'] != 'TOTAL'].set_index('age_start')
rows = []
for start in range(0, 60, 5):                       # 0-4 in 2017 -> 5-9 in 2022 ... 55-59 -> 60-64
    predicted = g.loc[start, 'y2017'] * group_survival(start, range(start, start + 5))
    rows.append({'2017 group': f'{start}-{start + 4}', '2022 group': f'{start + 5}-{start + 9}',
                 'predicted 2022': predicted, 'real 2022': g.loc[start + 5, 'y2022']})

# the open group 65+ in 2022 = survivors of 60-64 in 2017 + survivors of 65+ in 2017
w = un[(un['year'] == 2017) & (un['age_start'] >= 65)].groupby('age_start')['population'].sum()
w = w / w.sum()                                     # age mix inside 65+ (UN 2017)
surv_65plus = sum(w[a] * group_survival(a, range(a, a + 5) if a < 100 else [100]) for a in w.index)
pred_65 = g.loc[60, 'y2017'] * group_survival(60, range(60, 65)) + g.loc[65, 'y2017'] * surv_65plus
rows.append({'2017 group': '60-64 and 65+', '2022 group': '65+', 'predicted 2022': pred_65, 'real 2022': g.loc[65, 'y2022']})

backtest = pd.DataFrame(rows)
backtest['error %'] = (backtest['predicted 2022'] / backtest['real 2022'] - 1) * 100
print('\nBACKTEST - ageing 2017 forward to 2022 vs the real 2022 census population:')
print(backtest.round({'predicted 2022': 0, 'error %': 2}).to_string(index=False))

for _, row in backtest.iterrows():
    gate(abs(row['error %']) <= 3, f'backtest {row["2022 group"]}: predicted within ±3% of real ({row["error %"]:+.2f}%)')

backtest.to_csv(f'{CLEAN}/backtest_2017_to_2022.csv', index=False)
print(f'\nLargest error in any age group: {backtest["error %"].abs().max():.2f}%')
print(f'\nStep 2 complete: {len(GATE_LOG)} gates passed so far, 0 failed')

GATE PASSED: life table has 2,828 rows = 14 years x 2 sexes x 101 ages
GATE PASSED: every life table starts at 100,000 at birth
GATE PASSED: survivors only decrease with age, in every year and for both sexes
GATE PASSED: every one-year survival share is between 0 and 1 (2,828 values)

BACKTEST - ageing 2017 forward to 2022 vs the real 2022 census population:
   2017 group 2022 group  predicted 2022  real 2022  error %
          0-4        5-9       2229720.0    2257680    -1.24
          5-9      10-14       2066752.0    2100003    -1.58
        10-14      15-19       1847267.0    1858245    -0.59
        15-19      20-24       1720902.0    1719492     0.08
        20-24      25-29       1678534.0    1702262    -1.39
        25-29      30-34       1499608.0    1529157    -1.93
        30-34      35-39       1333899.0    1354051    -1.49
        35-39      40-44       1071786.0    1077993    -0.58
        40-44      45-49        787734.0     785913     0.23
        45-49      50-54     

## Step 3 – Age the 2022 Saudi population forward to 2030, one year at a time
Starting from the 2022 census (Saudi citizens, single ages, men and women separately), each year every person moves up one year of age, and only the share given by the life table survives:

> population(age + 1, next year) = population(age, this year) × survival(age, sex, this year)

This is repeated 8 times (2022 → 2030). People aged 100 and over stay in one open group (100+).

**Why births are not needed:** hip fractures in this study start at age 45, and everyone aged 45+ in 2030 was already alive in 2022 (aged 37+). Ages under 8 in 2030 are therefore not projected.

**Gates:**
- in each year, the projected population can only shrink (deaths only, no migration for citizens)
- every birth cohort shrinks every year
- the number of Saudis aged 65+ grows, as the population ages
- the saved table, read back, is identical

In [10]:
# Starting: I take the 2022 census for Saudi citizens and put everyone aged 100-110 into one 100+ group
base = pop[pop['nationality'] == 'saudi'].copy()
base['age'] = base['age'].clip(upper=100)
base = base.groupby(['sex', 'age'], as_index=False)['population'].sum()
gate(base['population'].sum() == 18_792_262, 'starting population = all 18,792,262 Saudi citizens in 2022')

# Now I age the population forward one year at a time, from 2022 to 2030
years = {2022: base.assign(year=2022)}
current = base.copy()
for year in range(2022, 2030):
    nxt = current.merge(surv[surv['year'] == year], on=['sex', 'age'], how='left')
    gate(nxt['survival'].notna().all(), f'{year}: every age and sex found its survival share')
    nxt['survivors'] = nxt['population'] * nxt['survival']
    nxt['age'] = (nxt['age'] + 1).clip(upper=100)            # everyone is one year older; 100+ stays 100+
    current = nxt.groupby(['sex', 'age'], as_index=False)['survivors'].sum().rename(columns={'survivors': 'population'})
    before, after = years[year]['population'].sum(), current['population'].sum()
    gate(after < before, f'{year} -> {year + 1}: population only shrinks through deaths ({before:,.0f} -> {after:,.0f})')
    years[year + 1] = current.assign(year=year + 1)

projection = pd.concat(years.values(), ignore_index=True)
projection = projection[projection['age'] >= projection['year'] - 2022]   # ages that existed in 2022 (no births added)

# Now I follow one birth cohort as a check: people aged 60 in 2022 must shrink every year until 2030
cohort = [projection[(projection['year'] == 2022 + k) & (projection['age'] == 60 + k)]['population'].sum() for k in range(9)]
gate(all(a > b for a, b in zip(cohort, cohort[1:])), 'the cohort aged 60 in 2022 shrinks every single year to 2030')

# Now I check that the older population grows as the population ages
def older(year, min_age):
    return projection[(projection['year'] == year) & (projection['age'] >= min_age)]['population'].sum()
gate(older(2030, 65) > older(2022, 65), f'Saudis aged 65+ grow: {older(2022, 65):,.0f} (2022) -> {older(2030, 65):,.0f} (2030)')

# Now I save the projection and read it back
out = f'{CLEAN}/saudi_population_projection_2022_2030.csv'
projection.to_csv(out, index=False)
check = pd.read_csv(out)
gate(len(check) == len(projection) and abs(check['population'].sum() - projection['population'].sum()) < 1e-3,
     f'projection saved and read back identical ({len(check):,} rows)')

# Finally I show how the older Saudi population changes between 2022 and 2030
summary = pd.DataFrame({
    'age group': ['50-64', '65-79', '80+', 'total 50+'],
    '2022': [older(2022, 50) - older(2022, 65), older(2022, 65) - older(2022, 80), older(2022, 80), older(2022, 50)],
    '2030': [older(2030, 50) - older(2030, 65), older(2030, 65) - older(2030, 80), older(2030, 80), older(2030, 50)],
})
summary['growth %'] = (summary['2030'] / summary['2022'] - 1) * 100
print('\nSaudi citizens by older age group:')
print(summary.round(0).to_string(index=False))
print(f'\nStep 3 complete: {len(GATE_LOG)} gates passed so far, 0 failed')

GATE PASSED: starting population = all 18,792,262 Saudi citizens in 2022
GATE PASSED: 2022: every age and sex found its survival share
GATE PASSED: 2022 -> 2023: population only shrinks through deaths (18,792,262 -> 18,736,825)
GATE PASSED: 2023: every age and sex found its survival share
GATE PASSED: 2023 -> 2024: population only shrinks through deaths (18,736,825 -> 18,688,725)
GATE PASSED: 2024: every age and sex found its survival share
GATE PASSED: 2024 -> 2025: population only shrinks through deaths (18,688,725 -> 18,640,959)
GATE PASSED: 2025: every age and sex found its survival share
GATE PASSED: 2025 -> 2026: population only shrinks through deaths (18,640,959 -> 18,592,405)
GATE PASSED: 2026: every age and sex found its survival share
GATE PASSED: 2026 -> 2027: population only shrinks through deaths (18,592,405 -> 18,542,847)
GATE PASSED: 2027: every age and sex found its survival share
GATE PASSED: 2027 -> 2028: population only shrinks through deaths (18,542,847 -> 18,492,19

## Step 4 – Hip fractures among Saudi citizens, 2022–2030, in three scenarios
The age- and sex-specific Saudi rates are multiplied by the projected population (ages 45+) for every year from 2022 to 2030.

| Scenario | Rates above age 80 | Meaning |
|---|---|---|
| **Low** | printed 80+ rate, flat | exactly what the paper's table shows |
| **Mid** | keep rising along the 70–80 trend **up to age 90**, flat after | follows the paper's statement that rates rise up to age 90 |
| **High** | Mid × calibration factor | calibrated so that 2015 reproduces the paper's published 2,949 |

**Refinement from Step 1:** the trend now stops at age 90 (the paper's own limit). Without this cap, the census's unusually large 100+ group (4,822 people – very old ages are often over-reported in censuses) received extreme rates. The calibration factor is recalculated with the same rule.

All three scenarios assume the **2017–18 rates stay the same** – the paper notes that rates could also rise, which would make the true numbers higher still.

**Gates:** recalculated calibration factor stays within 2% of Step 1 · low ≤ mid ≤ high in every year · men + women = total · fractures rise every year as the population ages · saved file read back identical.

In [11]:
# Starting: I write one rule that gives the rate for any single age, sex and scenario
R = {sex: rates[rates['sex'] == sex].set_index('age_start')['rate_per_100k'].to_dict() for sex in ['male', 'female']}
def rate_for(sex, age, method):
    start = min(age // 5 * 5, 90)                     # the trend stops at 90 (paper: rates rise up to age 90)
    if start < 80:
        return R[sex][start]
    if method == 'flat' or start == 80:
        return R[sex][80]
    slope = np.log(R[sex][80] / R[sex][70]) / 10      # yearly rise between the 70-74 and 80+ groups
    return R[sex][80] * np.exp(slope * (start - 80))

# Now I recalculate the calibration factor with the age-90 cap, exactly as in Step 1 (UN 2015, ages 50+)
un15 = un[(un['year'] == 2015) & (un['age_start'] >= 50)]
trend_2015 = sum(r.population * rate_for(r.sex, r.age_start, 'trend') / 100_000 for r in un15.itertuples())
old_cal = pd.read_csv(f'{CLEAN}/calibration_factor.csv')['calibration_factor'].iloc[0]
calibration = 2949 / trend_2015
gate(abs(calibration / old_cal - 1) < 0.02,
     f'calibration with the age-90 cap = {calibration:.3f}, within 2% of Step 1 ({old_cal:.3f})')
pd.DataFrame({'calibration_factor': [calibration], 'note': ['trend capped at age 90']}).to_csv(f'{CLEAN}/calibration_factor.csv', index=False)

# Now I apply the rates to the projected Saudi population aged 45+, for every year and scenario
p45 = projection[projection['age'] >= 45].copy()
p45['rate_low'] = [rate_for(s, a, 'flat')  for s, a in zip(p45['sex'], p45['age'])]
p45['rate_mid'] = [rate_for(s, a, 'trend') for s, a in zip(p45['sex'], p45['age'])]
p45['low']  = p45['population'] * p45['rate_low'] / 100_000
p45['mid']  = p45['population'] * p45['rate_mid'] / 100_000
p45['high'] = p45['mid'] * calibration

by_sex = p45.groupby(['year', 'sex'])[['low', 'mid', 'high']].sum()
total  = p45.groupby('year')[['low', 'mid', 'high']].sum()

# Now I check the results are consistent
gate(((total['low'] <= total['mid']) & (total['mid'] <= total['high'])).all(), 'low <= mid <= high in every year 2022-2030')
gate((abs(by_sex.groupby('year').sum() - total) < 1e-6).all().all(), 'men + women = total in every year and scenario')
for sc in ['low', 'mid', 'high']:
    gate(total[sc].is_monotonic_increasing, f'{sc} scenario: hip fractures rise every year as the population ages')

# Now I save the yearly results and read them back
results = by_sex.reset_index()
out = f'{CLEAN}/hip_fractures_2022_2030.csv'
results.to_csv(out, index=False)
gate(len(pd.read_csv(out)) == 9 * 2, 'yearly results saved and read back (9 years x 2 sexes)')

# Finally I show the headline numbers
show = total.round(0).astype(int)
show['growth vs 2022 (mid) %'] = ((total['mid'] / total.loc[2022, 'mid'] - 1) * 100).round(0).astype(int)
print('\nHip fractures per year among Saudi citizens aged 45+:')
print(show.to_string())
y30 = by_sex.loc[2030].round(0).astype(int)
print('\n2030 by sex:')
print(y30.to_string())
print(f"\n2030 headline: about {total.loc[2030, 'mid']:,.0f} hip fractures (range {total.loc[2030, 'low']:,.0f} - {total.loc[2030, 'high']:,.0f})")
print("For context only: the paper's own 2030 projection is 6,971 for ALL residents aged 50+ (UN 2019 population).")
print(f'\nStep 4 complete: {len(GATE_LOG)} gates passed so far, 0 failed')

GATE PASSED: calibration with the age-90 cap = 1.306, within 2% of Step 1 (1.297)
GATE PASSED: low <= mid <= high in every year 2022-2030
GATE PASSED: men + women = total in every year and scenario
GATE PASSED: low scenario: hip fractures rise every year as the population ages
GATE PASSED: mid scenario: hip fractures rise every year as the population ages
GATE PASSED: high scenario: hip fractures rise every year as the population ages
GATE PASSED: yearly results saved and read back (9 years x 2 sexes)

Hip fractures per year among Saudi citizens aged 45+:
       low   mid  high  growth vs 2022 (mid) %
year                                          
2022  2023  2282  2980                       0
2023  2093  2335  3049                       2
2024  2191  2429  3172                       6
2025  2299  2544  3322                      11
2026  2423  2671  3488                      17
2027  2559  2814  3675                      23
2028  2703  2967  3874                      30
2029  2869  314

## Step 5 – From fractures to implants: nails, hip replacements and screws
Each hip fracture is split by **site**, using the Saudi study's own patients: neck of femur 43%, pertrochanteric 47%, subtrochanteric 10%. Each site is then matched to its usual operation, following the UK NICE hip fracture guideline (CG124):

| Fracture site | Usual operation | Implant |
|---|---|---|
| Neck of femur, **displaced** | replace the femoral head | **hemiarthroplasty** (half hip) or **total hip replacement** for fitter patients |
| Neck of femur, **undisplaced** | fix in place | **screws** / sliding hip screw |
| Pertrochanteric | fix | **cephalomedullary nail** or **sliding hip screw** |
| Subtrochanteric | fix | **intramedullary nail** |

Saudi data on these choices is not published, so each share is an **assumption with a low–mid–high range**, stored in a table with its reasoning. The mid result uses mid fractures × mid shares; the range combines the low and high ends.

**Gates:** every share is between 0 and 1 · sites add up to 100% · in every scenario, the implants add up exactly to the operated fractures · low ≤ mid ≤ high for every implant type · saved files read back identical.

In [12]:
from itertools import product

# Starting: I load the Saudi fracture-site split from the paper
sites = pd.read_csv(f'{CLEAN}/hipfracture_sites.csv').set_index('fracture_site')['share']
gate(abs(sites.sum() - 1) < 1e-9, 'fracture sites add up to 100% (43 / 47 / 10)')

# Now I write down every treatment assumption with a low-mid-high range and the reason for it
assumptions = pd.DataFrame([
    ['operated',          0.90, 0.95, 0.98, 'Almost all hip fractures are operated; a few frail patients are treated without surgery'],
    ['neck_displaced',    0.70, 0.75, 0.80, 'Most femoral neck fractures in older people are displaced'],
    ['tha_of_displaced',  0.10, 0.20, 0.30, 'NICE: total hip replacement for fitter, independently walking patients; the rest get a hemiarthroplasty'],
    ['nail_of_pertroch',  0.50, 0.70, 0.90, 'Pertrochanteric fractures get a nail or a sliding hip screw; nails dominate in modern practice'],
    ['nail_of_subtroch',  1.00, 1.00, 1.00, 'NICE: subtrochanteric fractures are treated with an intramedullary nail'],
], columns=['assumption', 'low', 'mid', 'high', 'reason']).set_index('assumption')
gate(assumptions[['low', 'mid', 'high']].stack().between(0, 1).all(), 'every treatment share is between 0 and 1')
gate(((assumptions['low'] <= assumptions['mid']) & (assumptions['mid'] <= assumptions['high'])).all(), 'every assumption range is ordered low <= mid <= high')

# Now I write the rule that turns fractures into implants; it also checks that implants add up to operated fractures
def implants(fractures, a):
    operated = fractures * a['operated']
    neck, pertroch, subtroch = operated * sites['Neck of femur'], operated * sites['Pertrochanteric'], operated * sites['Subtrochanteric']
    out = pd.Series({
        'Hemiarthroplasty (half hip)': neck * a['neck_displaced'] * (1 - a['tha_of_displaced']),
        'Total hip replacement':       neck * a['neck_displaced'] * a['tha_of_displaced'],
        'Screws (undisplaced neck)':   neck * (1 - a['neck_displaced']),
        'Nail - pertrochanteric':      pertroch * a['nail_of_pertroch'],
        'Sliding hip screw':           pertroch * (1 - a['nail_of_pertroch']),
        'Nail - subtrochanteric':      subtroch * a['nail_of_subtroch'],
    })
    if abs(out.sum() - operated) > 1e-6:
        raise AssertionError('GATE FAILED: implants do not add up to operated fractures')
    out['ALL NAILS'] = out['Nail - pertrochanteric'] + out['Nail - subtrochanteric']
    out['ALL HIP REPLACEMENTS'] = out['Hemiarthroplasty (half hip)'] + out['Total hip replacement']
    return out

# Now I calculate the MID result: mid fractures x mid shares
mid = implants(total.loc[2030, 'mid'], assumptions['mid'])
mid22 = implants(total.loc[2022, 'mid'], assumptions['mid'])

# Now I test EVERY combination of low/high fractures and low/high shares (2 x 2^5 = 64) to find each implant's true range
names = list(assumptions.index)
combos = []
for f_end, *ends in product(['low', 'high'], *[['low', 'high']] * len(names)):
    shares = pd.Series({n: assumptions.loc[n, e] for n, e in zip(names, ends)})
    combos.append(implants(total.loc[2030, f_end], shares))
combos = pd.DataFrame(combos)
gate(len(combos) == 64, 'all 64 combinations calculated, and in each one implants add up to operated fractures')

imp = pd.DataFrame({'2022 (mid)': mid22, 'low': combos.min(), 'mid': mid, 'high': combos.max()})
gate(((imp['low'] <= imp['mid'] + 1e-9) & (imp['mid'] <= imp['high'] + 1e-9)).all(), 'for every implant type, the mid lies inside its low-high range')

# Now I save the assumptions and results and read them back
assumptions.to_csv(f'{CLEAN}/treatment_assumptions.csv')
imp.to_csv(f'{CLEAN}/implants_2030.csv')
gate(pd.read_csv(f'{CLEAN}/implants_2030.csv', index_col=0).shape == imp.shape, 'implant results saved and read back identical')

# Finally I show the implant demand
print('\nImplants needed for hip fractures in Saudi citizens:')
print(imp.round(0).astype(int).to_string())
print(f"\n2030 headline: about {imp.loc['ALL NAILS', 'mid']:,.0f} nails and {imp.loc['ALL HIP REPLACEMENTS', 'mid']:,.0f} hip replacements a year")
print(f'\nStep 5 complete: {len(GATE_LOG)} gates passed so far, 0 failed')

GATE PASSED: fracture sites add up to 100% (43 / 47 / 10)
GATE PASSED: every treatment share is between 0 and 1
GATE PASSED: every assumption range is ordered low <= mid <= high
GATE PASSED: all 64 combinations calculated, and in each one implants add up to operated fractures
GATE PASSED: for every implant type, the mid lies inside its low-high range
GATE PASSED: implant results saved and read back identical

Implants needed for hip fractures in Saudi citizens:
                             2022 (mid)  low   mid  high
Hemiarthroplasty (half hip)         559  577   816  1318
Total hip replacement               140   82   204   439
Screws (undisplaced neck)           233  235   340   549
Nail - pertrochanteric              713  643  1040  1801
Sliding hip screw                   306  129   446  1001
Nail - subtrochanteric              217  274   316   426
ALL NAILS                           930  917  1356  2227
ALL HIP REPLACEMENTS                699  824  1019  1465

2030 headline: about

## Step 6 – Rehabilitation demand, then the notebook gate
Every hip fracture patient needs rehabilitation, operated or not: physiotherapy in hospital from the day after surgery (NICE CG124), therapy after discharge, and a walking aid during recovery. These amounts vary by hospital and patient, so each is an **assumption with a low–mid–high range**, applied to the number of hip fractures. Because every assumption here increases demand, the low and high results come from combining all low or all high values.

### Notebook gate
The main result is **recalculated from scratch** with separate, independent code that reads only the **raw files** (census, UN life table and the paper's table), not the clean files or anything in memory. If the independent calculation does not match the saved 2030 result, the notebook stops. Other checks: the backtest stayed within ±3%, the calibration reproduces the published 2,949, and implants add up to operated fractures.

In [13]:
# Starting: I write down the rehabilitation assumptions per hip fracture patient, each with a range and reason
rehab = pd.DataFrame([
    ['inpatient_physio_sessions', 5, 7, 10,   'NICE: physiotherapy from the day after surgery, at least daily during the hospital stay'],
    ['post_discharge_sessions',   6, 12, 20,  'Outpatient or home physiotherapy after discharge to regain walking'],
    ['walker_share',              0.80, 0.90, 1.00, 'Most patients use a walking frame in the first weeks'],
], columns=['item', 'low', 'mid', 'high', 'reason']).set_index('item')
gate(((rehab['low'] <= rehab['mid']) & (rehab['mid'] <= rehab['high'])).all(), 'every rehab assumption range is ordered low <= mid <= high')

# Now I turn 2030 hip fractures into rehab demand (low x low, mid x mid, high x high: all assumptions push demand the same way)
rehab_2030 = {}
for sc in ['low', 'mid', 'high']:
    patients = total.loc[2030, sc]
    rehab_2030[sc] = {
        'Patients needing rehab': patients,
        'Physio sessions in hospital': patients * rehab.loc['inpatient_physio_sessions', sc],
        'Physio sessions after discharge': patients * rehab.loc['post_discharge_sessions', sc],
        'Walking frames': patients * rehab.loc['walker_share', sc],
    }
rehab_2030 = pd.DataFrame(rehab_2030)
rehab_2030.loc['Total physio sessions'] = rehab_2030.loc[['Physio sessions in hospital', 'Physio sessions after discharge']].sum()
gate(((rehab_2030['low'] <= rehab_2030['mid']) & (rehab_2030['mid'] <= rehab_2030['high'])).all(), 'rehab demand: low <= mid <= high for every item')
rehab.to_csv(f'{CLEAN}/rehab_assumptions.csv')
rehab_2030.to_csv(f'{CLEAN}/rehab_demand_2030.csv')
print('\nRehabilitation demand from hip fractures in Saudi citizens, 2030:')
print(rehab_2030.round(0).astype(int).to_string())

# ===================== NOTEBOOK GATE =====================
print('\n================ NOTEBOOK 2 GATE ================')
# Now I recalculate the 2030 mid result from the RAW files only, with separate code (arrays instead of tables)
raw_c = pd.read_csv(f'{RAW}/census2022_age_sex_nationality.csv', encoding='utf-8-sig')
raw_c = raw_c[(raw_c['Governorate Code'] == 'TOT_KSA') & (raw_c['Citizenship'] == 'Saudi')]
raw_l = pd.read_csv(f'{RAW}/un_wpp_saudi_lifetable_survivors_2017_2030.csv')
raw_r = pd.read_csv(f'{RAW}/manual_tables/hipfracture_rates_table1.csv')
raw_r = raw_r[raw_r['age_group'] != 'TOTAL']

independent = 0.0
for sex_raw, sex in [('Male', 'male'), ('Female', 'female')]:
    # population 2022 as an array of 101 ages (100 = 100+)
    people = np.zeros(101)
    part = raw_c[raw_c['Gender'] == sex_raw]
    np.add.at(people, np.minimum(part['Age'].str[1:].astype(int).values, 100), part['Observation value'].values)
    # age the array 8 times with the raw life table
    for year in range(2022, 2030):
        lx = raw_l[(raw_l['Time'] == year) & (raw_l['Sex'] == sex_raw)].sort_values('AgeStart')['Value'].values
        s = np.append(lx[1:] / lx[:-1], lx[100] / lx[99])
        new = np.zeros(101)
        new[1:] = people[:-1] * s[:-1]
        new[100] += people[100] * s[100]
        people = new
    # mid-scenario rates straight from the paper's table, trend above 80 capped at 90
    printed = dict(zip(raw_r['age_start'], raw_r[f'incidence_per_100000_{sex}']))
    slope = np.log(printed[80] / printed[70]) / 10
    for age in range(45, 101):
        start = min(age // 5 * 5, 90)
        r = printed[start] if start <= 80 else printed[80] * np.exp(slope * (start - 80))
        independent += people[age] * r / 100_000

saved_mid_2030 = pd.read_csv(f'{CLEAN}/hip_fractures_2022_2030.csv').query('year == 2030')['mid'].sum()
gate(abs(independent - saved_mid_2030) < 0.01,
     f'NB GATE: independent recalculation from raw files = {independent:,.2f}, saved result = {saved_mid_2030:,.2f}')

# Now the other end-of-notebook checks, all read fresh from Drive
bt = pd.read_csv(f'{CLEAN}/backtest_2017_to_2022.csv')
gate(bt['error %'].abs().max() <= 3, f'NB GATE: backtest - every age group within ±3% (largest {bt["error %"].abs().max():.2f}%)')
cal = pd.read_csv(f'{CLEAN}/calibration_factor.csv')['calibration_factor'].iloc[0]
gate(abs(trend_2015 * cal - 2949) < 0.01, 'NB GATE: calibration reproduces the published 2015 estimate of 2,949 exactly')
imp_saved = pd.read_csv(f'{CLEAN}/implants_2030.csv', index_col=0)
six = ['Hemiarthroplasty (half hip)', 'Total hip replacement', 'Screws (undisplaced neck)',
       'Nail - pertrochanteric', 'Sliding hip screw', 'Nail - subtrochanteric']
gate(abs(imp_saved.loc[six, 'mid'].sum() - saved_mid_2030 * 0.95) < 0.01,
     'NB GATE: mid implants add up to 95% of mid 2030 fractures (the operated share)')
res = pd.read_csv(f'{CLEAN}/hip_fractures_2022_2030.csv').groupby('year')[['low', 'mid', 'high']].sum()
gate(((res['low'] <= res['mid']) & (res['mid'] <= res['high'])).all(), 'NB GATE: saved yearly results keep low <= mid <= high')

# Finally the summary of Notebook 2
print(f"\nHEADLINE 2030: {res.loc[2030, 'mid']:,.0f} hip fractures (range {res.loc[2030, 'low']:,.0f}-{res.loc[2030, 'high']:,.0f}), "
      f"{imp_saved.loc['ALL NAILS', 'mid']:,.0f} nails, {imp_saved.loc['ALL HIP REPLACEMENTS', 'mid']:,.0f} hip replacements, "
      f"{rehab_2030.loc['Total physio sessions', 'mid']:,.0f} physio sessions")
print(f'\nNOTEBOOK 2 COMPLETE: {len(GATE_LOG)} gates passed, 0 failed. The 2030 result was recalculated independently from the raw files.')

GATE PASSED: every rehab assumption range is ordered low <= mid <= high
GATE PASSED: rehab demand: low <= mid <= high for every item

Rehabilitation demand from hip fractures in Saudi citizens, 2030:
                                   low    mid    high
Patients needing rehab            3042   3328    4345
Physio sessions in hospital      15208  23293   43454
Physio sessions after discharge  18250  39931   86908
Walking frames                    2433   2995    4345
Total physio sessions            33458  63223  130363

================ NOTEBOOK 2 GATE ================
GATE PASSED: NB GATE: independent recalculation from raw files = 3,327.55, saved result = 3,327.55
GATE PASSED: NB GATE: backtest - every age group within ±3% (largest 1.93%)
GATE PASSED: NB GATE: calibration reproduces the published 2015 estimate of 2,949 exactly
GATE PASSED: NB GATE: mid implants add up to 95% of mid 2030 fractures (the operated share)
GATE PASSED: NB GATE: saved yearly results keep low <= mid <= high

